# ReCTS recogniser: retrain, export, **and keep it**

Reviewer item 2 needs stock YOLOv11 and the paper's detector read by *the same*
fine-tuned recogniser. That recogniser no longer exists: it was exported to
`/kaggle/working/PP-OCRv5_server_rec_infer` but the archiving line in the original
notebook was commented out, so it went when the session was cleared.

This notebook rebuilds it and saves it. Nothing to attach — the data comes from Drive.

**Accelerator: GPU T4 x2. Runtime ~3 h.** Use *Save Version -> Save & Run All*, then
add this notebook's output as an input to the control notebook.

In [ ]:
PADDLE_ROOT = "/kaggle/working/PaddleOCR"
DATA_ROOT = "/kaggle/working/content/train_data/rec"
CONFIG = PADDLE_ROOT + "/configs/rec/PP-OCRv5/PP-OCRv5_mobile_rec.yml"
DICTIONARY = PADDLE_ROOT + "/configs/rec/multi_language/custom_reCTS_dict.txt"
PRETRAINED = PADDLE_ROOT + "/pretrained_models/PP-OCRv5_mobile_rec_pretrained.pdparams"
EXPORT_DIR = "/kaggle/working/PP-OCRv5_rects_rec_infer"
SAVE_MODEL_DIR = "/kaggle/working/rec_output"

# 98,253 crops at ~21 min/epoch on T4 x2. 35 epochs is ~12.2 h and does not fit
# Kaggle's 12 h cap - that is what killed the first attempt at epoch 34. The first
# run's log shows accuracy essentially flat from epoch 29 (0.748) to its best at 32
# (0.760), so 25 epochs costs about a point and finishes with hours to spare.
EPOCHS = 25
BATCH_SIZE = 64
GPUS = "0,1"

# Belt and braces: also publish the export as a standalone Kaggle Dataset.
# Needs KAGGLE_USERNAME and KAGGLE_KEY as notebook secrets.
PUBLISH_DATASET = True
DATASET_SLUG = "rishiksaisanthosh/rects-ppocrv5-finetuned"

In [ ]:
!pip install -q gdown
!git clone -q --branch ablation-mscbam-probe https://github.com/SaiSanthosh1508/End-to-End-Text-Translation-Pipeline.git /kaggle/working/repo
import sys; sys.path.insert(0, "/kaggle/working/repo")

## 1. ReCTS training images and line annotations

In [ ]:
!gdown -q 1orMtLhJt3rQl3pMoLm31eh-SmDG74W1K -O /kaggle/working/ReCTS.zip
!unzip -q -o /kaggle/working/ReCTS.zip -d /kaggle/working
!ls /kaggle/working | head

In [ ]:
from pathlib import Path
from rects_control.crops import build

train_n, val_n = build(Path("/kaggle/working/img"), Path("/kaggle/working/gt"), Path(DATA_ROOT))
print(f"{train_n} train crops, {val_n} val crops")
assert train_n > 10_000, "far fewer crops than expected - check the unzipped layout"

## 2. PaddleOCR and the PP-OCRv5 mobile checkpoint

In [ ]:
!git clone -q https://github.com/PaddlePaddle/PaddleOCR.git {PADDLE_ROOT}
!pip install -q paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/
!pip install -q -r {PADDLE_ROOT}/requirements.txt
!pip install -q lmdb rapidfuzz
!wget -q -P {PADDLE_ROOT}/pretrained_models https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_mobile_rec_pretrained.pdparams

In [ ]:
from rects_control.paddle_config import build_dictionary, patch_training_config

n_chars = build_dictionary(Path(DICTIONARY))
patch_training_config(
    Path(CONFIG), data_root=Path(DATA_ROOT), dictionary=Path(DICTIONARY),
    pretrained=Path(PRETRAINED), epochs=EPOCHS, batch_size=BATCH_SIZE,
    save_model_dir=Path(SAVE_MODEL_DIR),
)
print(f"{n_chars} characters in the label space")

## 3. Fine-tune

**~9 h on T4 x2.** Start it with a clear 12 h ahead of you.

Checkpoints go to an absolute `SAVE_MODEL_DIR`. The stock config uses a relative
`./output/...`, and that is what lost the first attempt: it trained for twelve hours,
logged `save model in ./output/PP-OCRv5_mobile_rec/latest` every epoch, and none of it
appeared in the session snapshot. Expect accuracy around 0.75 by epoch 25.

In [ ]:
!python3 -m paddle.distributed.launch --gpus '{GPUS}' {PADDLE_ROOT}/tools/train.py \
    -c {CONFIG} -o Global.pretrained_model={PRETRAINED}

## 4. Export for inference

Check the checkpoint is on disk before exporting. If this is empty the run produced
nothing, and no amount of exporting will conjure it back.

In [ ]:
checkpoints = sorted(Path(SAVE_MODEL_DIR).rglob("*.pdparams"))
for path in checkpoints:
    print(f"{path}  {path.stat().st_size // 1024} KB")
assert checkpoints, f"no checkpoint under {SAVE_MODEL_DIR} - training saved nothing"

best = Path(SAVE_MODEL_DIR) / "best_model" / "model.pdparams"
chosen = best if best.exists() else Path(SAVE_MODEL_DIR) / "latest.pdparams"
CHOSEN_STEM = str(chosen.with_suffix(""))   # PaddleOCR appends .pdparams itself
print("exporting from", CHOSEN_STEM)

In [ ]:
!python3 {PADDLE_ROOT}/tools/export_model.py -c {CONFIG} -o \
    Global.pretrained_model={CHOSEN_STEM} \
    Global.save_inference_dir={EXPORT_DIR}
!ls -la {EXPORT_DIR}

## 5. Persist it, then check it

Archiving comes before the quality gate on purpose. A Kaggle version that raises
saves no output at all, so an assertion here would throw away the two hours of
training it was meant to protect. Nothing below this line is allowed to raise.

In [ ]:
import shutil
archive = shutil.make_archive("/kaggle/working/rects_rec_finetuned", "zip", EXPORT_DIR)
size_kb = Path(archive).stat().st_size // 1024
print(archive, size_kb, "KB")
if size_kb < 1000:
    print("WARNING: archive is much smaller than a PP-OCRv5 mobile export should be")

### Optional: publish it as a Kaggle Dataset

The version output above is already permanent, and attaching it to notebook 2 is
enough. A dataset is the stronger option: it is independent of this notebook, so
editing or deleting the notebook cannot take the weights with it, and it is what you
would pull from for the Space.

Needs two notebook secrets - *Add-ons -> Secrets* - named `KAGGLE_USERNAME` and
`KAGGLE_KEY`, then set `PUBLISH_DATASET = True` above. Without them this cell says so
and moves on.

In [ ]:
import os, shutil, subprocess

if PUBLISH_DATASET:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")

    # Publish the export inside a named folder so the attached path is the same
    # shape as a notebook-output attachment, and notebook 2 finds either.
    stage = Path("/kaggle/working/recognizer_dataset")
    shutil.copytree(EXPORT_DIR, stage / Path(EXPORT_DIR).name, dirs_exist_ok=True)
    subprocess.run(
        ["python", "ablation/push_snapshot.py", "--dir", str(stage),
         "--slug", DATASET_SLUG, "--message", "ReCTS fine-tuned PP-OCRv5 mobile rec"],
        cwd="/kaggle/working/repo",
    )
else:
    print("PUBLISH_DATASET is False; the version output is your only copy")

### Does it actually read ReCTS text?

A broken export - wrong dictionary, wrong checkpoint - still loads and still returns
plausible strings. Reading held-out crops whose ground truth we know is the only cheap
way to tell, and it is worth knowing now rather than after the 4.4 h detector run.

This reports; it does not raise. Read the output before starting notebook 2.

In [ ]:
import cv2
from rects_control.recognizer import PaddleRecognizer

rows = Path(DATA_ROOT, "rec_gt_test.txt").read_text(encoding="utf-8").splitlines()[:12]
paths, truth = zip(*(r.split("\t") for r in rows))
try:
    predicted = PaddleRecognizer(Path(EXPORT_DIR))([cv2.imread(str(Path(DATA_ROOT, p))) for p in paths])
    hits = sum(p == t for p, t in zip(predicted, truth))
    for p, t in zip(predicted, truth):
        print(f"{'ok ' if p == t else '   '} pred={p!r:20s} gt={t!r}")
    verdict = "looks right" if hits >= 4 else "SUSPECT - do not start notebook 2 yet"
    print(f"\n{hits}/{len(truth)} exact on a 12-crop sample: {verdict}")
except Exception as error:                 # the weights are already archived above
    print(f"self-test could not run: {error!r}")
    print("The export is still saved; investigate before starting notebook 2.")

**Now: Save Version -> Save & Run All.**

When it finishes, open the control notebook and add this notebook's output under
*Add Input -> Notebook Output*. Also worth doing once: publish
`/kaggle/working/PP-OCRv5_rects_rec_infer` as a Kaggle dataset, so a cleared session
can never cost you these weights again.